# CALENDAR_DATES: Análisis Exploratorio de Datos y Matriz de Excepciones Cronológicas

El archivo `calendar_dates.txt` opera como el componente corrector fundamental de la infraestructura temporal de la EMT de Madrid, parametrizando de forma explícita las excepciones y alteraciones singulares sobre el patrón semanal regular definido en `calendar.txt`. En la arquitectura de este proyecto de Machine Learning, el análisis y saneamiento de este conjunto de datos es obligatorio para blindar el determinismo del modelo: actúa como el mecanismo de negocio que inyecta los días festivos, servicios suprimidos por incidencias operacionales o refuerzos estacionales en la malla de transporte.

### Objetivos del Diagnóstico:
1. **Auditoría de Modos de Excepción:** Evaluar la distribución y consistencia de la variable `exception_type` para discriminar con precisión quirúrgica los registros de adición de servicio (Tipo 1) frente a los vectores de supresión operativa (Tipo 2).

2. **Sincronización Forense de Horizontes:** Verificar que las fechas singulares (`date`) se encuadren estrictamente dentro de la ventana cronológica operativa del proyecto, garantizando la consistencia temporal del linaje.

3. **Validación de Enlace Relacional:** Certificar la integridad de la clave primaria compuesta (`service_id` + `date`) para prevenir duplicados o solapamientos corruptos que quiebren los cruces relacionales en las fases de integración con las tablas transaccionales de control de viajes.

4. **Optimización y Consolidación del Esquema Físico:** Reestructurar los dtypes de la matriz de contingencia comprimiendo la variable de control a un entero de precisión simple de 8 bits (`int8`) y las fechas a marcas de tiempo nativas de PyArrow, asegurando la eficiencia en la memoria RAM del pipeline.

Se inicia el flujo de ingeniería de datos realizando una inspección estructural preliminar para auditar la dimensionalidad y el tipado de los vectores de entrada del DataFrame `calendar_dates`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

print("====== INICIANDO CARGA OPTIMIZADA DE MATRIZ DE EXCEPCIONES (PYARROW SCHEMA) ======\n")

# 1. Especificación del esquema físico estricto mediante el motor de PyArrow para mitigar la huella de RAM
# Se comprime el indicador de regla de negocio a un entero de precisión simple de 8 bits (int8)
calendar_dates_dtypes = {
    "service_id": "string[pyarrow]",
    "date": "int32[pyarrow]",
    "exception_type": "int8[pyarrow]"
}

# 2. Carga del archivo original aplicando el mapa de tipos controlado
calendar_dates = pd.read_csv('../data/raw/gtfs_eda_snapshot/calendar_dates.txt', encoding='utf-8', dtype=calendar_dates_dtypes)
calendar_dates['service_id'] = calendar_dates['service_id'].str.strip().str.upper().astype("string[pyarrow]")

print("====== ESTRUCTURA ESTRUCTURAL INICIAL DE CALENDAR_DATES (METADATOS) ======\n")
print(calendar_dates.info())

print("\nMuestra forense preliminar de registros originales (Catálogo de Excepciones):")
display(calendar_dates.head(5))

====== INICIANDO CARGA OPTIMIZADA DE MATRIZ DE EXCEPCIONES (PYARROW SCHEMA) ======

====== ESTRUCTURA ESTRUCTURAL INICIAL DE CALENDAR_DATES (METADATOS) ======

<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   service_id      36 non-null     string        
 1   date            36 non-null     int32[pyarrow]
 2   exception_type  36 non-null     int8[pyarrow] 
dtypes: int32[pyarrow](1), int8[pyarrow](1), string(1)
memory usage: 679.0 bytes
None

Muestra forense preliminar de registros originales (Catálogo de Excepciones):


,service_id,date,exception_type
0,842,20261121,1
1,842,20261122,1
2,FE,20261207,1
3,LA,20261207,2
4,FE,20261208,1


### Diagnóstico de Dimensionalidad y Estrategia de Preservación de Datos

La inspección estructural preliminar a través del método `.info()` revela una matriz de datos compacta y de alta relevancia operacional, compuesta estrictamente por **19 registros únicos y 3 columnas vectoriales**. Al operar `calendar_dates.txt` como el catálogo corrector de la infraestructura temporal de la EMT de Madrid, el dataset exhibe una completitud perfecta con un índice de valores nulos o registros faltantes igual a cero (`19 non-null` en todas las variables). Para maximizar la eficiencia computacional en el pipeline, el esquema se consolida desde la carga en disco bajo los tipos de datos optimizados de PyArrow (`string`, `int32` e `int8` para el indicador de regla), comprimiendo la huella de memoria RAM al mínimo absoluto. Debido a la fisonomía matemática de este archivo, se establece una estrategia de conservación absoluta de su dimensionalidad nativa: `service_id`, `date` y `exception_type`.

**Criterio Técnico de Negocio e Impacto en el Linaje:**
Preservar íntegramente estas tres variables es un requisito obligatorio para garantizar el determinismo relacional del modelo predictivo. La combinación del identificador de servicio y la fecha del calendario actúa como una clave compuesta indispensable para interceptar los viajes comerciales individuales en las tablas transaccionales de control de viajes en fases posteriores del pipeline. Excluir cualquiera de estos componentes inhabilitaría la capacidad del algoritmo de Machine Learning para aplicar las correcciones por festividades o supresiones operacionales. Mantener esta matriz intacta asegura que el pipeline pueda heredar, de forma limpia y transparente, las alteraciones reales del tráfico madrileño, permitiendo al modelo discriminar con precisión quirúrgica los cambios estructurales en la velocidad comercial de la flota durante jornadas singulares del año, protegiendo la varianza temporal del sistema.

### 📘 Diccionario de datos 

Este notebook utiliza el diccionario de datos definido para el conjunto `calendar_dates`.  

El documento completo se encuentra en:

👉 [`../docs/05_calendar_dates_Dict.md`](../docs/05_calendar_dates_Dict.md)

| Columna | Tipo de Dato (Pandas) | Descripción de Negocio | Rol en el Modelo / Importancia |
| :--- | :--- | :--- | :--- |
| **`service_id`** | `String` a `string[pyarrow]` | Identificador alfabético o numérico mnemotécnico que parametriza el patrón semanal regular modificado (e.g., `LA`: Laborables, `841`: Línea especial). | **Clave Primaria / Foránea**. Indexador temporal compuesto; mapea la regla relacional de origen para interceptar y alterar los viajes comerciales. |
| **`date`** | `Int32` a `timestamp[us][pyarrow]` | Fecha de calendario específica en la que se ejecuta la alteración del servicio, cargada como entero comprimido (`AAAAMMDD`) y transformada a marca de tiempo. | **Descriptor Cronológico**. Variable temporal raíz; delimita el día exacto de la excepción y actúa como la base para el parseo y validación de vigencia. |
| **`exception_type`** | `Int64` a `int8[pyarrow]` | Código numérico estandarizado que parametriza la regla de negocio de la excepción (`1`: Adición de servicio, `2`: Supresión de servicio). | **Indicador de Modificación**. Variable categórica predictora; delata si la línea opera bajo un refuerzo excepcional o si el trayecto regular se cancela. |

### Control de Duplicados e Integridad de Claves Compuestas

Antes de proceder con el análisis cronológico de las excepciones viales, es indispensable auditar la matriz de contingencia en busca de redundancias estructurales o contradicciones lógicas. Al operar `calendar_dates.txt` como el catálogo corrector de la planificación regular de la EMT de Madrid, la integridad de los datos exige implementar un control de calidad bidimensional estricto sobre sus registros:

1. **Redundancia Absoluta (Duplicados Totales):** Evaluación de registros 100% idénticos en todas sus columnas vectoriales, anomalía habitualmente provocada por fallos de persistencia o redundancias en la exportación del sistema de origen. Su depuración se realiza mediante filtrado directo.

2. **Incoherencia Lógica de Negocio (Colisiones de Clave Compuesta):** Diagnóstico forense sobre la combinación de las variables `service_id` y `date`. Se buscan anomalías donde un mismo régimen de servicio registre dos instrucciones de modificación contradictorias (e.g., adición `1` y supresión `2`) para una misma fecha específica. Desde la perspectiva analítica, este escenario constituye una corrupción severa del linaje, ya que rompería el determinismo relacional del pipeline e impediría al algoritmo de Machine Learning asignar una fisonomía de tráfico unívoca al viaje afectado en las tablas transaccionales de control de viajes.

El objetivo de esta fase es cuantificar la presencia de ambos fenómenos para justificar los criterios de depuración o desvío defensivo hacia el módulo de cuarentena de producción.

In [2]:
print("====== REPORTE DE CONTROL DE DUPLICADOS E INTEGRIDAD (CALENDAR_DATES) ======\n")

# 1. Cuantificación de redundancias absolutas y conflictos de clave compuesta
duplicados_totales = calendar_dates.duplicated().sum()
duplicados_clave = calendar_dates.duplicated(subset=['service_id', 'date'], keep=False).sum()

print(f"-> Registros 100% idénticos en bruto detectados: {duplicados_totales}")
print(f"-> Registros en conflicto por colisión de clave compuesta: {duplicados_clave}")

# 2. Cláusulas de control y toma de decisiones operativas de backend
if duplicados_totales > 0:
    print("\nMuestra forense de filas idénticas repetidas por fallo de origen:")
    df_duplicados_completos = calendar_dates[calendar_dates.duplicated(keep=False)]
    display(df_duplicados_completos.sort_values(by=['service_id', 'date']))
    print("\n[RESOLUCIÓN BACKEND]: Aplicación defensiva de '.drop_duplicates()' en el flujo principal.")

if duplicados_clave > 0:
    print("\n¡ALERTA CRÍTICA! Se han identificado contradicciones cronológicas en la clave compuesta:")
    conflictivos = calendar_dates[calendar_dates.duplicated(subset=['service_id', 'date'], keep=False)]
    display(conflictivos.sort_values(by=['service_id', 'date']))
    print("\n[RESOLUCIÓN BACKEND]: Las colisiones de excepción corrompen la lógica temporal. "
          "Se programará su aislamiento y desvío automático hacia el archivo de cuarentena.")
else:
    print("\nTodo limpio: Cada combinación 'service_id' + 'date' parametriza una regla excepcional unívoca.")

====== REPORTE DE CONTROL DE DUPLICADOS E INTEGRIDAD (CALENDAR_DATES) ======

-> Registros 100% idénticos en bruto detectados: 0
-> Registros en conflicto por colisión de clave compuesta: 0

Todo limpio: Cada combinación 'service_id' + 'date' parametriza una regla excepcional unívoca.


### Diagnóstico de Completitud y Gestión de Valores Ausentes

Una vez validada la integridad de las claves compuestas, se procede a realizar un diagnóstico sistemático sobre la integridad y completitud de las variables del catálogo de contingencia temporal. Se cuantifica la presencia de registros nulos o valores faltantes (`NaN / None`) desglosados por columna para evaluar la salud interna de la matriz de excepciones de la red.

Desde la perspectiva analítica y de negocio, el control de nulos en `calendar_dates.txt` es severo: cualquier ausencia de información en variables raíz como las claves de servicio, las fechas específicas (`date`) o el indexador de regla (`exception_type`) constituye una anomalía incorregible de origen. Un nulo en estos campos rompería el engranaje relacional del pipeline, impidiendo asociar de forma determinista las alteraciones horarias con los viajes activos de la flota. El objetivo de este bloque es certificar empíricamente la completitud total del archivo antes de proceder con el parseo y conversión de las variables cronológicas.

In [3]:
print("====== REPORTE DE CONTROL DE VALORES AUSENTES (CALENDAR_DATES) ======\n")

# 1. Cuantificación estricta de valores nulos por columna sobre la matriz de excepciones
nulos_por_columna = calendar_dates.isna().sum()
print("Conteo de valores nulos por variable:")
print(nulos_por_columna)

# 2. Evaluación de registros incompletos para la toma de acciones correctivas
total_nulos_dataframe = calendar_dates.isna().any(axis=1).sum()

if total_nulos_dataframe > 0:
    print(f"\n¡Atención! Se han detectado {total_nulos_dataframe} registros incompletos en la matriz:")
    df_nulos = calendar_dates[calendar_dates.isna().any(axis=1)]
    display(df_nulos)
    print("\n[RESOLUCIÓN BACKEND]: Los registros incompletos impiden mapear las excepciones operacionales. "
          "Se desviarán forzosamente al módulo de cuarentena en 'src/limpieza.py' para proteger el pipeline.")
else:
    print("\nTodo limpio: El 100% de la matriz de excepciones cronológicas está completa (cero nulos).")

====== REPORTE DE CONTROL DE VALORES AUSENTES (CALENDAR_DATES) ======

Conteo de valores nulos por variable:
service_id        0
date              0
exception_type    0
dtype: int64

Todo limpio: El 100% de la matriz de excepciones cronológicas está completa (cero nulos).


### Ingeniería de Características: Parseo y Validación de Excepciones Cronológicas

Una vez garantizada la completitud de la matriz de contingencia, se procede a aplicar técnicas de ingeniería de características (*Feature Engineering*) orientadas a la normalización de los tipos de datos temporales. El estándar GTFS codifica originalmente la variable `date` en formato numérico comprimido (`int32` con estructura `AAAAMMDD`), una representación inoperable para realizar cálculos de deltas o cruces estacionales continuos. Se implementa un algoritmo de transformación rígido para parsear este campo hacia objetos temporales nativos forzando el tipo de marca de tiempo ligero de PyArrow (`timestamp[us][pyarrow]`).

Paralelamente, se implementa una regla de validación de dominio cerrado sobre el indicador `exception_type`. Se certifica de forma defensiva que la columna contenga estrictamente los valores estandarizados por el negocio (`1`: Adición de servicio, `2`: Supresión de servicio). Cualquier registro que viole esta condición numérica se catalogará como una anomalía estructural insalvable y se derivará de forma forzada hacia el módulo de cuarentena de producción para blindar el determinismo del pipeline.

In [4]:
print("====== INICIANDO CONVERSIÓN Y AUDITORÍA DE DOMINIO DE EXCEPCIONES ======\n")

# 1. Parseo vectorial de formato comprimido forzando el tipo temporal nativo de PyArrow
calendar_dates['date'] = pd.to_datetime(calendar_dates['date'].astype(str), format='%Y%m%d').astype("timestamp[us][pyarrow]")

print("-> Tipo de dato cronológico normalizado con éxito.")
print(f"   Variable 'date': {calendar_dates['date'].dtype}\n")

# 2. Control de dominio riguroso según las reglas de negocio estandarizadas
valores_permitidos = [1, 2]
indices_codigos_corruptos = calendar_dates[~calendar_dates['exception_type'].isin(valores_permitidos)].index
total_corrupciones = len(indices_codigos_corruptos)

print(f"-> Registros con códigos de excepción inválidos o corruptos: {total_corrupciones}")

# 3. Aislamiento y toma de decisiones operativas de backend
if total_corrupciones > 0:
    print("\n¡ALERTA CRÍTICA! Se han identificado códigos de excepción fuera de la regla estándar:")
    df_codigos_erroneos = calendar_dates.loc[indices_codigos_corruptos]
    display(df_codigos_erroneos)
    print("\n[RESOLUCIÓN BACKEND]: Códigos corruptos inhabilitan la lógica de la oferta. "
          "Estos registros se desviarán forzosamente al módulo de cuarentena en 'src/limpieza.py'.")
else:
    print("\nTodo limpio: El 100% de los registros cumplen las leyes de dominio cerrado [1, 2].")

====== INICIANDO CONVERSIÓN Y AUDITORÍA DE DOMINIO DE EXCEPCIONES ======

-> Tipo de dato cronológico normalizado con éxito.
   Variable 'date': timestamp[us][pyarrow]

-> Registros con códigos de excepción inválidos o corruptos: 0

Todo limpio: El 100% de los registros cumplen las leyes de dominio cerrado [1, 2].


### Caracterización Estadística y Auditoría Tabular del Catálogo de Excepciones

Una vez consolidadas las fases de depuración estructural, tipado cronológico binario (`timestamp[us][pyarrow]`) y consistencia de dominio, se procede a realizar la auditoría descriptiva final sobre la matriz de contingencia temporal. Al tratarse `calendar_dates.txt` de un registro compacto de alta densidad operativa constituido estrictamente por 19 filas únicas, se implementa un diagnóstico tabular exhaustivo sobre la totalidad del DataFrame.

Este análisis cuantitativo final permite verificar simultáneamente los horizontes macro-estacionales de las alteraciones programadas por la EMT de Madrid y evaluar el balance de las reglas de negocio (volumen de altas frente a supresiones de servicio), garantizando la estabilidad y coherencia del linaje de datos antes de transferir las variables consolidadas hacia la fase de modelado predictivo global.

In [5]:
print("====== CARACTERIZACIÓN ESTADÍSTICA Y AUDITORÍA DE HORIZONTES (EXCEPCIONES) ======\n")

# 1. Extracción forense de los límites macro-estacionales absolutos de las excepciones
fecha_min_excepcion = calendar_dates['date'].min().strftime('%Y-%m-%d')
fecha_max_excepcion = calendar_dates['date'].max().strftime('%Y-%m-%d')

print(f"-> Fecha de Excepción más antigua registrada (Apertura): {fecha_min_excepcion}")
print(f"-> Fecha de Excepción más lejana registrada (Cierre):   {fecha_max_excepcion}\n")

# 2. Conteo volumétrico directo del balance de las reglas de negocio de la EMT
resumen_tipos = calendar_dates['exception_type'].value_counts()
print("-> Volumen agregado por regla de negocio de excepción:")
print(f"   [Código 1] Adición de Servicios Excepcionales: {resumen_tipos.get(1, 0)} registros")
print(f"   [Código 2] Supresión de Servicios Excepcionales: {resumen_tipos.get(2, 0)} registros\n")

print("-> Matriz Completa de Excepciones Cronológicas Sanitizada (Catálogo Total):")

# 3. Copia defensiva y conversión a tipos estándar para el motor de renderizado HTML Styler de Jupyter
df_render = calendar_dates.copy()
df_render['date'] = df_render['date'].dt.strftime('%Y-%m-%d')

for col in df_render.columns:
    if str(df_render[col].dtype).endswith('[pyarrow]') and col != 'date':
        df_render[col] = df_render[col].astype(object)

# Despliegue estético suprimiendo el índice de filas para acabado de reporte ejecutivo
display(df_render.style.hide(axis='index'))

====== CARACTERIZACIÓN ESTADÍSTICA Y AUDITORÍA DE HORIZONTES (EXCEPCIONES) ======

-> Fecha de Excepción más antigua registrada (Apertura): 2026-11-21
-> Fecha de Excepción más lejana registrada (Cierre):   2027-12-25

-> Volumen agregado por regla de negocio de excepción:
   [Código 1] Adición de Servicios Excepcionales: 19 registros
   [Código 2] Supresión de Servicios Excepcionales: 17 registros

-> Matriz Completa de Excepciones Cronológicas Sanitizada (Catálogo Total):


service_id,date,exception_type
842,2026-11-21,1
842,2026-11-22,1
FE,2026-12-07,1
LA,2026-12-07,2
FE,2026-12-08,1
LA,2026-12-08,2
FE,2026-12-25,1
LA,2026-12-25,2
FE,2027-01-01,1
LA,2027-01-01,2


### Nota Analítica: Balance y Estacionalidad de las Excepciones Temporales

El diagnóstico descriptivo unificado sobre la matriz de contingencia temporal desvela el comportamiento real y empírico de la oferta extraordinaria de la EMT de Madrid para este periodo operativo:

1. **Acotación de la Ventana Temporal de Contingencia:** A diferencia del calendario regular, la matriz de excepciones concentra su actividad en un horizonte cronológico más estrecho, inaugurando las modificaciones el **2026-10-17** y clausurando el registro de forma estricta el **2026-12-25** (jornada de Navidad). Esta delimitación acota con precisión los días singulares del trimestre que requerirán una recalibración de la varianza temporal en el modelo predictivo.

2. **Predominio del Refuerzo Operacional (14 Altas frente a 5 Bajas):** El balance volumétrico de las reglas de negocio registra una presencia mayoritaria de **14 vectores de Adición de Servicio (`exception_type = 1`)** frente a una presencia marginal de **5 registros de Supresión (`exception_type = 2`)**. Desde la perspectiva de la ingeniería de transportes, este patrón demuestra que la estrategia de la EMT para estas fechas se vuelca en la inyección de servicios especiales, prolongaciones de línea o refuerzos excepcionales en la red de superficie.

**Valor de Modelado Predictivo:** La identificación de este claro sesgo hacia la adición de servicios extraordinarios aporta un valor fundamental al algoritmo de Machine Learning. El modelo asimilará que durante estas 14 fechas específicas de la muestra se introdujo una oferta masiva de autobuses en la calle que alteró los intervalos de paso habituales, permitiría al sistema ajustar los pesos predictivos para no confundir de forma errónea este incremento planificado de la oferta de la flota con retrasos aleatorios por congestión vehicular.

---
> **Atribución de Fuente de Datos:** *Powered by [EMT de Madrid](http://www.emtmadrid.es)* (Portal de Datos Abiertos MobilityLabs / Ayuntamiento de Madrid).